In [2]:
import numpy as np
import pandas as pd

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
df = pd.read_csv("covid.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   age        100 non-null    int64  
 1   gender     100 non-null    object 
 2   fever      90 non-null     float64
 3   cough      100 non-null    object 
 4   city       100 non-null    object 
 5   has_covid  100 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 4.8+ KB


In [7]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=["has_covid"]), df["has_covid"], test_size=0.2
)

In [9]:
X_train

,age,gender,fever,cough,city
65,69,Female,102.0,Mild,Bangalore
85,16,Female,103.0,Mild,Bangalore
78,11,Male,100.0,Mild,Bangalore
12,25,Female,99.0,Strong,Kolkata
20,12,Male,98.0,Strong,Bangalore
...,...,...,...,...,...
55,81,Female,101.0,Mild,Mumbai
53,83,Male,98.0,Mild,Delhi
62,56,Female,104.0,Strong,Bangalore
92,82,Female,102.0,Strong,Kolkata


In [10]:
X_test

,age,gender,fever,cough,city
49,44,Male,104.0,Mild,Mumbai
61,81,Female,98.0,Strong,Mumbai
58,23,Male,98.0,Strong,Mumbai
89,46,Male,103.0,Strong,Bangalore
84,69,Female,98.0,Strong,Mumbai
16,69,Female,103.0,Mild,Kolkata
82,24,Male,98.0,Mild,Kolkata
19,42,Female,NaN,Strong,Bangalore
11,65,Female,98.0,Mild,Mumbai
22,71,Female,98.0,Strong,Kolkata


# Before Ski-Learn & Traditional using Pandas

In [11]:
# adding simple imputer to fever col
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])
                                 
X_train_fever.shape

(80, 1)

In [12]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [14]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

In [15]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

In [16]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

# After Ski-Learn using Sklearn-ColumnTransformer

In [17]:
from sklearn.compose import ColumnTransformer

In [19]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

In [20]:
transformer.fit_transform(X_train).shape

(80, 7)

In [21]:
transformer.transform(X_test).shape

(20, 7)